<a href="https://colab.research.google.com/github/PrathamVashisht26/Real-Time-Social-Media-Sentiment-Topic-Analysis-Engine-/blob/main/Social_media_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-dotenv

In [ ]:
!pip install emoji nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 1.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import tweepy
import pandas as pd
import os

bearer_token = userdata.get("BEARER_TOKEN")

if not bearer_token:
  raise Exception("Twitter bearer token not found! Did you upload the.env file correctly?")

client = tweepy.Client(bearer_token)
print("Autherntication successful!")

search_query = '"Electric vehicles" -is:retweet lang:en'
print(f" searching for: {search_query}")

try:
  response = client.search_recent_tweets(
      query = search_query,
      max_results = 100,
      tweet_fields = ["created_at", "text", "lang"]
  )

  tweets = response.data

  if not tweets:
    print("No tweets found for this query.")
  else:
    print(f"Successfully fetched {len(tweets)} tweets.")

    tweet_data = [
        {"id": tweet.id, "created_at": tweet.created_at, "text": tweet.text}
        for tweet in tweets
    ]

    df = pd.DataFrame(tweet_data)

    df.to_csv("raw_tweets.csv", index = False)

    print("\n DataFrame created and saved to 'raw_tweets.csv'")
    print("\n--- First 5 Tweets ---")
    print(df.head())

except Exception as e:
  print(f"An error occured: {e}")

Autherntication successful!
 searching for: "Electric vehicles" -is:retweet lang:en
An error occured: 429 Too Many Requests
Usage cap exceeded: Monthly product cap


In [ ]:
import pandas as pd
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

print("Advanced libraries installed and ready")

print("\n Loading raw_tweets.csv")
try:
  df = pd.read_csv("raw_tweets.csv")
except FileNotFoundError:
  print("Error: raw_tweets.csv not found. Please make sure you have run step 2 correctly.")
  raise

def advanced_cleaned_text(text):

  lemmatizer = WordNetLemmatizer()
  stop_words = set(stopwords.words('english'))

  text = emoji.demojize(text)

  text = re.sub(r'https?://\S+|www\.S', '' ,text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'#', '', text)

  text = re.sub(r'_', ' ', text)

  text = re.sub(r'[^a-zA-Z\s]', '', text)

  tokens = text.lower().split()

  cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

  return " ".join(cleaned_tokens)

print("performing advanced cleaning. . .")
df['advanced_cleaned_text'] = df['text'].apply(advanced_cleaning_text)
print("Advanced Cleaning completed")

df.to_csv("cleaned_tweets_advanced.csv", index=False)
print("\n Advanced cleaned data saved to ' cleaned_tweets_advanced.csv'")
print("\n---Comparing Original vs. Advanced Cleaned Text ---")

print(df[['text', 'advanced_cleaned_text']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Advanced libraries installed and ready

 Loading raw_tweets.csv
Error: raw_tweets.csv not found. Please make sure you have run step 2 correctly.


FileNotFoundError: [Errno 2] No such file or directory: 'raw_tweets.csv'

In [ ]:
!pip install -q google-generativeai
import pandas as pd
from google.colab import userdata
import google.generativeai as genai
import json
import time
print("✅ Libraries installed.")
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configured successfully!")
except Exception as e:
    print("❌ Error configuring Gemini API. Make sure your secret is named 'GEMINI_API_KEY'.")
    raise
try:
    df = pd.read_csv("cleaned_tweets_advanced.csv")
    df = df.head(20)
    print(f"✅ Loaded {len(df)} cleaned tweets from 'cleaned_tweets_advanced.csv'.")
except FileNotFoundError:
    print("❌ Error: 'cleaned_tweets_advanced.csv' not found. Please run Step 3 first.")
    raise
model = genai.GenerativeModel('gemini-pro')
def analyze_tweet(tweet_text):
    prompt = f"""
    Analyze the sentiment and the main topic of the following tweet.
    The sentiment must be one of: Positive, Negative, or Neutral.
    The topic should be a short, 1-3 word phrase.
    Return the result as a JSON object with two keys: "sentiment" and "topic".
    Tweet: "{tweet_text}"
    JSON Output:
    """
    try:
        response = model.generate_content(prompt)
        json_response = response.text.strip().replace('`', '').replace('json', '')
        result = json.loads(json_response)
        return result
    except Exception as e:
        print(f"  - Could not analyze tweet. Error: {e}")
        return None
results = []
for index, row in df.iterrows():
    print(f"Processing tweet {index + 1}/{len(df)}...")
    tweet = row['advanced_cleaned_text']
    if not tweet:
        results.append({'sentiment': 'N/A', 'topic': 'N/A'})
        continue
    analysis_result = analyze_tweet(tweet)
    if analysis_result:
        results.append(analysis_result)
    else:
        results.append({'sentiment': 'Error', 'topic': 'Error'})
    time.sleep(1)
results_df = pd.DataFrame(results)
df['sentiment'] = results_df['sentiment']
df['topic'] = results_df['topic']
df.to_csv("analyzed_tweets.csv", index=False)
print("\n✅ Analysis complete! Results saved to 'analyzed_tweets.csv'")
print("\n--- Final Analyzed Data ---")
print(df[['advanced_cleaned_text', 'sentiment', 'topic']].head())


SyntaxError: incomplete input (ipython-input-2742640497.py, line 15)